# 03. Avaliar linkage

Duas perguntas, a mesma pares 1:1 da `cohort_dedup` ∩ subset (lista toda confiável). A ouro é **amostra**,
não o universo; a `cohort_dedup` é toda confiável — só N:1 / 1:N saem em `n_nao_1a1_descartada`.
Lista operacional (1 CPF por Censo) fica no [`04_atribuir.ipynb`](04_atribuir.ipynb).

| Seção | Unidade | Capa | Não é |
|-------|---------|------|-------|
| A — score | par Censo×CPF | `cobertura_blocking`, `recall_par_ouro`, `fp_amostra` | precision populacional |
| B — cluster | Censo A | funil, `recall_cluster_ouro` por tipo | o mesmo recall da seção A |

- **cobertura_blocking:** ouro 1:1 do subset que colide em ≥1 regra. Miss de blocking.
- **recall_par_ouro:** ouro 1:1 com par em `splink_predictions` e `match_probability ≥ THRESHOLD_AVALIACAO`. Sem linha no parquet = miss de blocking (já na cobertura).
- **fp_amostra:** fração da amostra de pares distintos conhecidos (até 5/âncora, mesmas regras) com score ≥ threshold. Negativos *difíceis*, não precision.
- **recall_cluster_ouro:** o cluster de A contém o CPF ouro X. Mega-cluster N×M infla — ver quebra por tipo.

Amostra de erros (miss de blocking, FN de score, FP da amostra) e sweep de threshold
(`metricas_sweep.csv`) vêm depois da capa A. O sweep é SQL no parquet; cluster
só no `THRESHOLD_AVALIACAO` — transitividade pode recuperar ouro que o proxy de par
não conta.

Curva P/R do Splink e waterfall são diagnóstico da amostra rotulada, não número de capa.

**Pré-requisito:** `02b_aplicar` (`splink_predictions.parquet`, `splink_clusters.parquet`) e `censo_limpo`/`cpf_limpo`. JSON do modelo só para o diagnóstico opcional.


In [ ]:
import json
import sys
from pathlib import Path

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

import pandas as pd
from IPython.display import display

from config import (
    COHORT_DEDUP_ARQUIVO,
    METRICAS_AVALIACAO,
    METRICAS_SWEEP,
    SPLINK_CLUSTERS,
    SPLINK_INPUT_VIEW,
    SPLINK_MODEL_JSON,
    SPLINK_PREDICTIONS,
    TABELA_CENSO_LIMPA,
    TABELA_CPF_LIMPA,
    THRESHOLD_AVALIACAO,
    drop_splink_temp_tables,
    get_connection,
    get_splink_db_api,
    materialize_cluster_composicao,
    materialize_gt_no_subset,
    materialize_splink_input,
    print_paths,
    require_input,
    require_tables,
)
from splink_spec import blocking_rules_or_sql

N_PARES_DISTINTOS_POR_ANCORA = 5
TOP_N_CLUSTERS = 10
TOP_N_ERROS = 30
SWEEP_THRESHOLDS = (0.50, 0.80, 0.90, 0.95, 0.98, 0.99)

print_paths()
require_input(COHORT_DEDUP_ARQUIVO, label='COHORT')
require_input(SPLINK_PREDICTIONS, label='SPLINK_PREDICTIONS (rode o 02b_aplicar antes)')
require_input(SPLINK_CLUSTERS, label='SPLINK_CLUSTERS (rode o 02b_aplicar antes)')

con = get_connection()
drop_splink_temp_tables(con)
require_tables(con, [TABELA_CENSO_LIMPA, TABELA_CPF_LIMPA], notebook_origem='00b')
materialize_splink_input(con)
print('Threshold:', THRESHOLD_AVALIACAO)


## 1. Ouro 1:1 no subset

Só pares estritamente 1:1 com os dois lados no recorte. Cada linha é um Censo A
que deveria receber o CPF X.


In [ ]:
counts = materialize_gt_no_subset(con, cohort_parquet=COHORT_DEDUP_ARQUIVO)
n_gt = counts['n_gt_no_subset']
print('Pares coorte nacional:', f"{counts['n_pares_coorte_nacional']:,}")
print('1:1 nacional:', f"{counts['n_pares_1a1_nacional']:,}")
print('Não 1:1 descartados (N:1 / 1:N):', f"{counts['n_nao_1a1_descartada']:,}")
print('Ouro 1:1 no subset:', f'{n_gt:,}')
if n_gt == 0:
    raise RuntimeError(
        'Nenhum par da coorte caiu no subset — confira o filtro geográfico do NB00.'
    )


## A. Qualidade do score (pares)

Métricas de capa no parquet do `02b`. O P/R do Splink mais abaixo é a mesma
amostra rotulada, não o universo.


In [ ]:
block_sql = blocking_rules_or_sql('ca', 'pb')

con.execute(f'''
CREATE OR REPLACE TABLE splink_predictions AS
SELECT
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_l
        ELSE unique_id_r
    END AS unique_id_l,
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_r
        ELSE unique_id_l
    END AS unique_id_r,
    match_probability
FROM read_parquet('{SPLINK_PREDICTIONS}')
''')

capa_a = con.execute(f'''
SELECT
    COUNT(*) AS n_ouro,
    SUM(CASE WHEN {block_sql} THEN 1 ELSE 0 END) AS n_ouro_no_blocking,
    ROUND(
        100.0 * SUM(CASE WHEN {block_sql} THEN 1 ELSE 0 END)
        / NULLIF(COUNT(*), 0),
        2
    ) AS cobertura_blocking,
    SUM(CASE WHEN p.match_probability >= {THRESHOLD_AVALIACAO} THEN 1 ELSE 0 END)
        AS n_recall_par_ouro,
    ROUND(
        1.0 * SUM(CASE WHEN p.match_probability >= {THRESHOLD_AVALIACAO} THEN 1 ELSE 0 END)
        / NULLIF(COUNT(*), 0),
        4
    ) AS recall_par_ouro
FROM gt_no_subset gt
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = gt.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = gt.unique_id_cpf
LEFT JOIN splink_predictions p
  ON p.unique_id_l = gt.unique_id_censo AND p.unique_id_r = gt.unique_id_cpf
''').df()
display(capa_a)


Pares distintos conhecidos: Censo de A × CPF de B (A ≠ B na ouro), nas mesmas
regras de blocking. Quem não passa em nenhuma regra nunca é pontuado.

Até 5 por âncora (`N_PARES_DISTINTOS_POR_ANCORA`). `fp_amostra` não é precision.


In [ ]:
con.execute(f'''
CREATE OR REPLACE TABLE pares_distintos AS
SELECT
    source_dataset_l, unique_id_l, source_dataset_r, unique_id_r
FROM (
    SELECT
        'censo' AS source_dataset_l,
        ga.unique_id_censo AS unique_id_l,
        'cpf' AS source_dataset_r,
        gb.unique_id_cpf AS unique_id_r,
        row_number() OVER (PARTITION BY ga.unique_id_censo ORDER BY random()) AS rn
    FROM gt_no_subset ga
    JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = ga.unique_id_censo
    JOIN gt_no_subset gb ON gb.person_id_censo <> ga.person_id_censo
    JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = gb.unique_id_cpf
    WHERE ({blocking_rules_or_sql("ca", "pb")})
)
WHERE rn <= {N_PARES_DISTINTOS_POR_ANCORA}
''')

fp = con.execute(f'''
SELECT
    COUNT(*) AS n_distintos_amostra,
    SUM(CASE WHEN p.match_probability >= {THRESHOLD_AVALIACAO} THEN 1 ELSE 0 END)
        AS n_fp_amostra,
    ROUND(
        1.0 * SUM(CASE WHEN p.match_probability >= {THRESHOLD_AVALIACAO} THEN 1 ELSE 0 END)
        / NULLIF(COUNT(*), 0),
        4
    ) AS fp_amostra
FROM pares_distintos d
LEFT JOIN splink_predictions p
  ON p.unique_id_l = d.unique_id_l AND p.unique_id_r = d.unique_id_r
''').df()
display(fp)

n_distintos = int(fp.n_distintos_amostra.iloc[0])
if n_distintos == 0:
    print(
        'AVISO: nenhum par distinto conhecido gerado. Com poucos pares no subset '
        'não há base para fp_amostra — amplie o recorte geográfico.'
    )


### Amostra de erros

Três tabelas (`LIMIT` = `TOP_N_ERROS`). Join ouro / pares distintos + `splink_input`
+ parquet. Nome, DOB, UF, CEP e score.

1. **Miss de blocking** — ouro 1:1 que não colide em nenhuma regra. Não entra no parquet.
2. **FN de score** — ouro no blocking, mas `match_probability < THRESHOLD_AVALIACAO` ou
   ausente do parquet (`predict` só grava ≥ 0,5).
3. **FP amostra** — `pares_distintos` com score ≥ threshold (negativos difíceis, não
   precision populacional).


In [ ]:
block_sql = blocking_rules_or_sql('ca', 'pb')
cols_par = f'''
    ca.nome_completo AS nome_censo,
    pb.nome_completo AS nome_cpf,
    ca.data_nascimento AS dob_censo,
    pb.data_nascimento AS dob_cpf,
    ca.uf AS uf_censo,
    pb.uf AS uf_cpf,
    ca.cep AS cep_censo,
    pb.cep AS cep_cpf,
    p.match_probability
'''

n_miss = con.execute(f'''
SELECT COUNT(*)
FROM gt_no_subset gt
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = gt.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = gt.unique_id_cpf
WHERE NOT ({block_sql})
''').fetchone()[0]
print(f'Miss de blocking (ouro 1:1): {n_miss:,}  (amostra {TOP_N_ERROS})')
display(con.execute(f'''
SELECT
    gt.unique_id_censo,
    gt.unique_id_cpf,
    {cols_par}
FROM gt_no_subset gt
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = gt.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = gt.unique_id_cpf
LEFT JOIN splink_predictions p
  ON p.unique_id_l = gt.unique_id_censo AND p.unique_id_r = gt.unique_id_cpf
WHERE NOT ({block_sql})
LIMIT {TOP_N_ERROS}
''').df())

n_fn = con.execute(f'''
SELECT COUNT(*)
FROM gt_no_subset gt
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = gt.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = gt.unique_id_cpf
LEFT JOIN splink_predictions p
  ON p.unique_id_l = gt.unique_id_censo AND p.unique_id_r = gt.unique_id_cpf
WHERE ({block_sql})
  AND (p.match_probability IS NULL OR p.match_probability < {THRESHOLD_AVALIACAO})
''').fetchone()[0]
print(
    f'FN de score (no blocking, score < {THRESHOLD_AVALIACAO} '
    f'ou ausente do parquet): {n_fn:,}  (amostra {TOP_N_ERROS})'
)
display(con.execute(f'''
SELECT
    gt.unique_id_censo,
    gt.unique_id_cpf,
    CASE
        WHEN p.match_probability IS NULL THEN 'ausente_parquet'
        ELSE 'score_abaixo_do_corte'
    END AS motivo_fn,
    {cols_par}
FROM gt_no_subset gt
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = gt.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = gt.unique_id_cpf
LEFT JOIN splink_predictions p
  ON p.unique_id_l = gt.unique_id_censo AND p.unique_id_r = gt.unique_id_cpf
WHERE ({block_sql})
  AND (p.match_probability IS NULL OR p.match_probability < {THRESHOLD_AVALIACAO})
ORDER BY COALESCE(p.match_probability, -1) DESC
LIMIT {TOP_N_ERROS}
''').df())

n_fp_tab = con.execute(f'''
SELECT COUNT(*)
FROM pares_distintos d
JOIN splink_predictions p
  ON p.unique_id_l = d.unique_id_l AND p.unique_id_r = d.unique_id_r
WHERE p.match_probability >= {THRESHOLD_AVALIACAO}
''').fetchone()[0]
print(
    f'FP amostra (pares distintos com score ≥ {THRESHOLD_AVALIACAO}): '
    f'{n_fp_tab:,}  (amostra {TOP_N_ERROS})'
)
display(con.execute(f'''
SELECT
    d.unique_id_l AS unique_id_censo,
    d.unique_id_r AS unique_id_cpf,
    {cols_par}
FROM pares_distintos d
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = d.unique_id_l
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = d.unique_id_r
JOIN splink_predictions p
  ON p.unique_id_l = d.unique_id_l AND p.unique_id_r = d.unique_id_r
WHERE p.match_probability >= {THRESHOLD_AVALIACAO}
ORDER BY p.match_probability DESC
LIMIT {TOP_N_ERROS}
''').df())


### Sweep de threshold

Grid SQL no parquet (`SWEEP_THRESHOLDS`). `recall_par_ouro(t)`, `fp_amostra(t)` e
`n_censo_com_par(t)` = Censos com ≥1 CPF pontuado ≥ t. Cluster operacional
(`recall_cluster_ouro`) **não** entra no grid — reclustering Splink em vários cortes
é caro; o cluster ainda pode recuperar ouro via transitividade, o que o recall de
par ignora. Use a seção B para o corte `THRESHOLD_AVALIACAO`.

Arquivo separado (`metricas_sweep.csv`), não um pivot da linha única de capa.


In [ ]:
grid_sql = ', '.join(str(float(t)) for t in SWEEP_THRESHOLDS)
sweep = con.execute(f'''
WITH grid AS (
    SELECT unnest([{grid_sql}]) AS threshold
),
recall AS (
    SELECT
        g.threshold,
        ROUND(
            1.0 * SUM(CASE WHEN p.match_probability >= g.threshold THEN 1 ELSE 0 END)
            / NULLIF(COUNT(*), 0),
            4
        ) AS recall_par_ouro
    FROM grid g
    CROSS JOIN gt_no_subset gt
    LEFT JOIN splink_predictions p
      ON p.unique_id_l = gt.unique_id_censo AND p.unique_id_r = gt.unique_id_cpf
    GROUP BY 1
),
fp_t AS (
    SELECT
        g.threshold,
        ROUND(
            1.0 * SUM(CASE WHEN p.match_probability >= g.threshold THEN 1 ELSE 0 END)
            / NULLIF(COUNT(*), 0),
            4
        ) AS fp_amostra
    FROM grid g
    CROSS JOIN pares_distintos d
    LEFT JOIN splink_predictions p
      ON p.unique_id_l = d.unique_id_l AND p.unique_id_r = d.unique_id_r
    GROUP BY 1
),
censo_t AS (
    SELECT
        g.threshold,
        COUNT(DISTINCT p.unique_id_l) AS n_censo_com_par
    FROM grid g
    JOIN splink_predictions p ON p.match_probability >= g.threshold
    GROUP BY 1
)
SELECT
    r.threshold,
    r.recall_par_ouro,
    f.fp_amostra,
    COALESCE(c.n_censo_com_par, 0) AS n_censo_com_par,
    (abs(r.threshold - {THRESHOLD_AVALIACAO}) < 1e-12) AS corte_operacional
FROM recall r
LEFT JOIN fp_t f USING (threshold)
LEFT JOIN censo_t c USING (threshold)
ORDER BY r.threshold
''').df()
display(sweep)
sweep.to_csv(METRICAS_SWEEP, index=False)
print('Sweep salvo:', METRICAS_SWEEP)


### Diagnóstico (amostra rotulada, não população)

Labels Splink (`clerical_match_score` é o nome que a API exige — não é revisão
clerical). Curva P/R e waterfall dos erros no `THRESHOLD_AVALIACAO`. FN aqui
exclui ouro que não passou no blocking (já medido em `cobertura_blocking`).


In [ ]:
require_input(SPLINK_MODEL_JSON, label='SPLINK_MODEL_JSON (rode o 02_treinar antes)')

con.execute('''
CREATE OR REPLACE TABLE splink_labels AS
SELECT
    'censo' AS source_dataset_l,
    unique_id_censo AS unique_id_l,
    'cpf' AS source_dataset_r,
    unique_id_cpf AS unique_id_r,
    1.0 AS clerical_match_score
FROM gt_no_subset
UNION
SELECT
    source_dataset_l, unique_id_l, source_dataset_r, unique_id_r,
    0.0 AS clerical_match_score
FROM pares_distintos
''')
df_labels = con.execute('SELECT * FROM splink_labels').df()
display(con.execute('''
SELECT
    CASE clerical_match_score
        WHEN 1.0 THEN 'match conhecido (ouro)'
        WHEN 0.0 THEN 'par distinto conhecido'
    END AS tipo,
    COUNT(*) AS n
FROM splink_labels GROUP BY 1 ORDER BY tipo
''').df())


In [ ]:
from splink import Linker

db_api = get_splink_db_api(con)
with open(SPLINK_MODEL_JSON, 'r') as file:
    data = json.load(file)
data['retain_intermediate_calculation_columns'] = True

con.execute(f'''
CREATE OR REPLACE VIEW splink_censo AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'censo'
''')
con.execute(f'''
CREATE OR REPLACE VIEW splink_cpf AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'cpf'
''')

linker = Linker(
    ['splink_censo', 'splink_cpf'],
    data,
    db_api=db_api,
    input_table_aliases=['censo', 'cpf'],
)
labels_sdf = linker.table_management.register_labels_table(df_labels, overwrite=True)
print(f'Labels registradas: {len(df_labels):,}')


In [ ]:
linker.evaluation.accuracy_analysis_from_labels_table(
    labels_sdf,
    output_type='accuracy',
    match_weight_round_to_nearest=0.02,
)


In [ ]:
records_fp = linker.evaluation.prediction_errors_from_labels_table(
    labels_sdf,
    threshold_match_probability=THRESHOLD_AVALIACAO,
    include_false_negatives=False,
    include_false_positives=True,
).as_record_dict(limit=20)
print('Falsos positivos (amostra):', len(records_fp))
if records_fp:
    display(linker.visualisations.waterfall_chart(records_fp, filter_nulls=False))


In [ ]:
records_fn = linker.evaluation.prediction_errors_from_labels_table(
    labels_sdf,
    threshold_match_probability=THRESHOLD_AVALIACAO,
    include_false_negatives=True,
    include_false_positives=False,
).as_record_dict(limit=20)
print('Falsos negativos (amostra):', len(records_fn))
if records_fn:
    display(linker.visualisations.waterfall_chart(records_fn, filter_nulls=False))


## B. Recall operacional (clusters)

Acerto: o cluster de A contém X. Outros Censos no mesmo cluster são ok.
`1_para_1` e `1_cpf_n_censo` são os casos naturais. `outros` (N CPFs × M Censos)
inflaciona `recall_cluster_ouro`.


In [ ]:
con.execute(f'''
CREATE OR REPLACE TABLE splink_clusters AS
SELECT * FROM read_parquet('{SPLINK_CLUSTERS}')
''')

materialize_cluster_composicao(con)

display(con.execute('''
SELECT tipo, COUNT(*) AS n_clusters, SUM(n) AS n_registros,
       SUM(n_censo) AS n_censo, SUM(n_cpf) AS n_cpf
FROM cluster_composicao
GROUP BY tipo
ORDER BY n_clusters DESC
''').df())


In [ ]:
funil = con.execute(f'''
WITH censo_all AS (
    SELECT unique_id
    FROM {SPLINK_INPUT_VIEW}
    WHERE origem = 'censo'
),
censo_cluster AS (
    SELECT c.unique_id, sc.cluster_id, comp.tipo, comp.n_cpf
    FROM censo_all c
    LEFT JOIN splink_clusters sc ON sc.unique_id = c.unique_id
    LEFT JOIN cluster_composicao comp ON comp.cluster_id = sc.cluster_id
),
linkados AS (
    SELECT unique_id
    FROM censo_cluster
    WHERE n_cpf >= 1
),
ouro_hit AS (
    SELECT
        gt.unique_id_censo,
        CASE WHEN cl.cluster_id IS NOT NULL AND cl.cluster_id = cr.cluster_id
             THEN 1 ELSE 0 END AS hit,
        COALESCE(comp.tipo, 'sem_cluster') AS tipo_cluster
    FROM gt_no_subset gt
    LEFT JOIN splink_clusters cl ON cl.unique_id = gt.unique_id_censo
    LEFT JOIN splink_clusters cr ON cr.unique_id = gt.unique_id_cpf
    LEFT JOIN cluster_composicao comp ON comp.cluster_id = cl.cluster_id
)
SELECT
    (SELECT COUNT(*) FROM censo_all) AS n_censo_subset,
    (SELECT COUNT(*) FROM linkados) AS n_censo_com_match,
    ROUND(
        100.0 * (SELECT COUNT(*) FROM linkados)
        / NULLIF((SELECT COUNT(*) FROM censo_all), 0),
        2
    ) AS pct_censo_linkado,
    (SELECT COUNT(*) FROM gt_no_subset) AS n_ouro_cluster,
    (SELECT SUM(hit) FROM ouro_hit) AS n_ouro_recuperado,
    ROUND(
        1.0 * (SELECT SUM(hit) FROM ouro_hit)
        / NULLIF((SELECT COUNT(*) FROM gt_no_subset), 0),
        4
    ) AS recall_cluster_ouro,
    (
        SELECT COUNT(*)
        FROM linkados l
        JOIN gt_no_subset g ON g.unique_id_censo = l.unique_id
    ) AS n_linkados_no_ouro
''').df()
funil['threshold_avaliacao'] = THRESHOLD_AVALIACAO
display(funil)

print(
    f"Dos {int(funil.n_censo_subset.iloc[0]):,} Censos no subset, "
    f"{int(funil.n_censo_com_match.iloc[0]):,} ({funil.pct_censo_linkado.iloc[0]}%) "
    f"caíram num cluster com ≥1 CPF.\n"
    f"Ouro no subset: {int(funil.n_ouro_cluster.iloc[0]):,}; recuperados: "
    f"{int(funil.n_ouro_recuperado.iloc[0]):,} "
    f"→ recall_cluster_ouro = {funil.recall_cluster_ouro.iloc[0]}"
)


Recall por tipo de cluster. Se `outros` concentrar os hits, o agregado está inflado.


In [ ]:
display(con.execute('''
SELECT
    COALESCE(comp.tipo, 'sem_cluster') AS tipo_cluster,
    COUNT(*) AS n_censo_ouro,
    SUM(CASE WHEN cl.cluster_id = cr.cluster_id THEN 1 ELSE 0 END) AS hits,
    ROUND(
        1.0 * SUM(CASE WHEN cl.cluster_id = cr.cluster_id THEN 1 ELSE 0 END)
        / NULLIF(COUNT(*), 0),
        4
    ) AS recall_cluster_ouro
FROM gt_no_subset gt
LEFT JOIN splink_clusters cl ON cl.unique_id = gt.unique_id_censo
LEFT JOIN splink_clusters cr ON cr.unique_id = gt.unique_id_cpf
LEFT JOIN cluster_composicao comp ON comp.cluster_id = cl.cluster_id
GROUP BY 1
ORDER BY n_censo_ouro DESC
''').df())


Top clusters — homônimos, CEP compartilhado ou over-clustering.


In [ ]:
top = con.execute(f'''
WITH top_ids AS (
    SELECT cluster_id, n, tipo, n_censo, n_cpf
    FROM cluster_composicao
    ORDER BY n DESC
    LIMIT {TOP_N_CLUSTERS}
)
SELECT
    t.cluster_id, t.n, t.tipo, t.n_censo, t.n_cpf,
    s.unique_id, s.origem, s.nome_completo, s.data_nascimento, s.sexo, s.cep
FROM top_ids t
JOIN splink_clusters sc ON sc.cluster_id = t.cluster_id
JOIN {SPLINK_INPUT_VIEW} s ON s.unique_id = sc.unique_id
ORDER BY t.n DESC, t.cluster_id, s.origem, s.unique_id
''').df()
display(
    top.groupby(['cluster_id', 'n', 'tipo', 'n_censo', 'n_cpf'])
    .size().rename('membros').reset_index()
    .sort_values('n', ascending=False)
)
for cid, g in top.groupby('cluster_id', sort=False):
    display(g.drop(columns=['n', 'tipo', 'n_censo', 'n_cpf']).head(30))


In [ ]:
metricas = capa_a.copy()
metricas['n_distintos_amostra'] = fp['n_distintos_amostra'].iloc[0]
metricas['n_fp_amostra'] = fp['n_fp_amostra'].iloc[0]
metricas['fp_amostra'] = fp['fp_amostra'].iloc[0]
for col in funil.columns:
    metricas[col] = funil[col].iloc[0]
metricas['n_nao_1a1_descartada'] = counts['n_nao_1a1_descartada']
metricas['n_pares_coorte_nacional'] = counts['n_pares_coorte_nacional']
metricas.to_csv(METRICAS_AVALIACAO, index=False)
print('Métricas salvas:', METRICAS_AVALIACAO)
print('Sweep (já gravado):', METRICAS_SWEEP)
display(metricas.T)
con.close()
